# Train MobileNetThis notebook trains the MobileNet model for recycling classification.

In [ ]:
import sysimport torchimport torch.optim as optimfrom torch.utils.data import DataLoaderfrom pathlib import Path# Add project root to pathproject_root = Path('..').resolve()if str(project_root) not in sys.path:    sys.path.append(str(project_root))from src.utils.config import Configfrom src.data.dataset import RecyclingDatasetfrom src.data.augmentation import get_train_transforms, get_val_transformsfrom src.models.mobilenet import MobileNetClassifier, export_to_onnx, export_to_torchscriptfrom src.training.trainer import Trainerfrom src.training.losses import get_loss_functionfrom src.training.callbacks import ModelCheckpoint, EarlyStoppingfrom src.utils.visualization import plot_training_history, show_batch

## 1. Setup

In [ ]:
# Load configconfig = Config('../configs/mobilenet_config.yaml')# Set devicedevice = 'cuda' if torch.cuda.is_available() else 'cpu'print(f"Using device: {device}")# Create directoriesPath('../outputs/checkpoints').mkdir(parents=True, exist_ok=True)Path('../outputs/logs').mkdir(parents=True, exist_ok=True)Path('../outputs/exports').mkdir(parents=True, exist_ok=True)

## 2. Data Loading

In [ ]:
# Transformsimg_size = config.data.get('image_size', 224)train_transforms = get_train_transforms(img_size)val_transforms = get_val_transforms(img_size)# Datasetsprocessed_dir = Path('../') / config.data.get('processed_dir')train_dataset = RecyclingDataset(processed_dir / 'train.csv', root_dir='../data', transforms=train_transforms)val_dataset = RecyclingDataset(processed_dir / 'val.csv', root_dir='../data', transforms=val_transforms)print(f"Train size: {len(train_dataset)}")print(f"Val size: {len(val_dataset)}")print(f"Classes: {train_dataset.class2idx}")# Dataloadersbatch_size = config.training.get('batch_size', 32)num_workers = config.training.get('num_workers', 2)train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)

## 3. Visualize Batch

In [ ]:
# Show sample batchimages, labels = next(iter(train_loader))show_batch(images, labels, train_dataset.idx2class)

## 4. Model Initialization

In [ ]:
model = MobileNetClassifier(    num_classes=config.model.get('num_classes'),    architecture=config.model.get('architecture'),    pretrained=config.model.get('pretrained'),    dropout=config.model.get('dropout'))print(f"Model created: {config.model.get('architecture')}")

## 5. Training Setup

In [ ]:
# Loss and Optimizercriterion = get_loss_function(config.training, device)optimizer = optim.Adam(    model.parameters(),     lr=float(config.training.get('lr')),     weight_decay=float(config.training.get('weight_decay')))# Schedulerscheduler = optim.lr_scheduler.CosineAnnealingLR(    optimizer,     T_max=config.training.get('epochs'))# Callbackscallbacks = [    ModelCheckpoint(dirpath='../outputs/checkpoints', monitor='val_f1_macro', mode='max'),    EarlyStopping(monitor='val_loss', patience=config.training.get('early_stopping', {}).get('patience', 5))]# Trainertrainer = Trainer(    model=model,    optimizer=optimizer,    criterion=criterion,    device=device,    config=config._config,    scheduler=scheduler,    callbacks=callbacks)

## 6. Train

In [ ]:
epochs = config.training.get('epochs', 10)history = trainer.fit(train_loader, val_loader, epochs=epochs)

## 7. Results

In [ ]:
plot_training_history(history)

## 8. Export Model

In [ ]:
# Load best modelbest_model_path = '../outputs/checkpoints/best_model.pt'checkpoint = torch.load(best_model_path)model.load_state_dict(checkpoint['model_state_dict'])# Exportexport_to_onnx(model, '../outputs/exports/mobilenet.onnx')export_to_torchscript(model, '../outputs/exports/mobilenet_ts.pt')